# Data Preprocessing into H3 Tiles for TrajLearn

In [93]:
import pandas as pd
import numpy as np
import h3

In [94]:
RES = [6, 7, 8, 9, 10]

In [95]:
def df_to_h3(df, dataset, h3_resolution):
    df = df.set_index('timestamp')

    # Group by lion, resample to 1-hour ('1h') frequencies, and forward-fill (ffill) missing hours
    df = df.groupby('lion_id')[['latitude', 'longitude']].resample('1h').ffill().bfill().reset_index()

    def get_h3_index(row):
        try:
            return h3.latlng_to_cell(row['latitude'], row['longitude'], h3_resolution)
        except AttributeError:
            return h3.geo_to_h3(row['latitude'], row['longitude'], h3_resolution)

    df['h3_index'] = df.apply(get_h3_index, axis=1)

    df = df.groupby('lion_id').agg(
        date=('timestamp', lambda x: x.min().date()),
        higher_order_trajectory=('h3_index', lambda x: ' '.join(x))
    ).reset_index()

    df['trip_id'] = 1

    df = df[['lion_id', 'date', 'trip_id', 'higher_order_trajectory']]
    
    df.to_csv(f'{dataset}_res{h3_resolution}.csv', index=False)
    
    return df

In [ ]:
def df_to_h3_sliding(df, dataset, h3_resolution, window_size, stride):
    df = df.set_index('timestamp')

    # Group by lion, resample to 1-hour ('1h') frequencies, and forward-fill (ffill) missing hours
    df = df.groupby('lion_id')[['latitude', 'longitude']].resample('1h').ffill().bfill().reset_index()

    def get_h3_index(row):
        try:
            return h3.latlng_to_cell(row['latitude'], row['longitude'], h3_resolution)
        except AttributeError:
            return h3.geo_to_h3(row['latitude'], row['longitude'], h3_resolution)

    df['h3_index'] = df.apply(get_h3_index, axis=1)

    windowed_data = []

    # Process each lion's timeline individually
    for lion_id, group in df.groupby('lion_id'):
        group = group.sort_values('timestamp')
        
        h3_sequence = group['h3_index'].tolist()
        time_sequence = group['timestamp'].tolist()
        
        total_pings = len(h3_sequence)
        trip_id = 1
        
        # Slide the window across the sequence
        for i in range(0, total_pings - window_size + 1, stride):
            window_h3 = h3_sequence[i : i + window_size]
            
            start_date = time_sequence[i].date()
            
            windowed_data.append({
                'lion_id': lion_id,
                'date': start_date,
                'trip_id': trip_id,
                'higher_order_trajectory': ' '.join(window_h3)
            })
            
            trip_id += 1

    df = pd.DataFrame(windowed_data)
    
    df.to_csv(f'{dataset}_res{h3_resolution}_w{window_size}_s{stride}.csv', index=False)
    
    return df

## Botswana

In [96]:
botswana = pd.read_csv('/Users/connoryablonski/Dropbox/Gaia-AI/lstm/botswana_lion_fixes.csv')
botswana = botswana.rename(columns={
    'individual-local-identifier': 'lion_id',
    'height-above-ellipsoid': 'height',
    'location-lat': 'latitude',
    'location-long': 'longitude',
    'external-temperature': 'temperature',
    'ground-speed': 'speed'
})
botswana['timestamp'] = pd.to_datetime(botswana['timestamp'])
botswana = botswana.sort_values(by=['lion_id', 'timestamp'])

In [97]:
botswana.head()

,event-id,visible,timestamp,longitude,latitude,algorithm-marked-outlier,temperature,speed,height,manually-marked-outlier,sensor-type,individual-taxon-canonical-name,tag-local-identifier,lion_id,study-name
141482,33097521055,True,2009-11-26 00:00:00,22.736218,-21.364617,NaN,0.0,0.000000,1048.0,NaN,gps,Panthera leo,AL154,BF053,African lions in Central Kalahari Botswana
141483,33097521056,True,2009-11-26 01:00:00,22.736235,-21.364621,NaN,0.0,0.000548,1051.0,NaN,gps,Panthera leo,AL154,BF053,African lions in Central Kalahari Botswana
141484,33097521047,True,2009-11-26 02:00:00,22.736159,-21.364297,NaN,0.0,0.010209,1050.0,NaN,gps,Panthera leo,AL154,BF053,African lions in Central Kalahari Botswana
141485,33097521044,True,2009-11-26 03:00:00,22.736147,-21.364288,NaN,0.0,0.000456,1048.0,NaN,gps,Panthera leo,AL154,BF053,African lions in Central Kalahari Botswana
141486,33097521049,True,2009-11-26 04:00:00,22.736162,-21.364256,NaN,0.0,0.001197,1044.0,NaN,gps,Panthera leo,AL154,BF053,African lions in Central Kalahari Botswana


In [ ]:
for res in RES:
    df_to_h3(botswana, 'botswana', res)
    df_to_h3_sliding(botswana, 'botswana', res, window_size, stride)

## Kenya

In [99]:
jan_data = pd.read_excel('Lion fixes data_Jan28_2025.xlsx')
feb_data = pd.read_csv('Feb1_Mar9_lion fixes.csv', encoding='cp1252')
feb_data = feb_data.rename(columns={
    'Index': 'No',
    'Tag': 'Lion ID',
    'Speed': 'Li'
})
feb_data['Time Stamp'] = pd.to_datetime(feb_data['Time Stamp'])
raw_data = pd.concat([jan_data, feb_data], ignore_index=True)

raw_data['Latitude'] = raw_data['Latitude'].str.replace('[^\d\.-]', '', regex=True).astype(float)
raw_data['Longitude'] = raw_data['Longitude'].str.replace('[^\d\.-]', '', regex=True).astype(float)
raw_data['Lion ID'] = raw_data['Lion ID'].str.extract('(\d+)').astype(int)
raw_data = raw_data.rename(columns={
    'Lion ID': 'lion_id',
    'Time Stamp': 'timestamp',
    'Latitude': 'latitude',
    'Longitude': 'longitude'
})
kenya = raw_data[['lion_id', 'timestamp', 'latitude', 'longitude']].copy()

kenya['timestamp'] = pd.to_datetime(kenya['timestamp'])
kenya = kenya.sort_values(by=['lion_id', 'timestamp'])
kenya = kenya.drop_duplicates(subset=['lion_id', 'timestamp'], keep='first')

In [100]:
kenya.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7252 entries, 0 to 7255
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   lion_id    7252 non-null   int64         
 1   timestamp  7252 non-null   datetime64[ns]
 2   latitude   7252 non-null   float64       
 3   longitude  7252 non-null   float64       
dtypes: datetime64[ns](1), float64(2), int64(1)
memory usage: 283.3 KB


In [101]:
for res in RES:
    df_to_h3(kenya, 'kenya', res)